# AILA 2019 Legal Dataset Preparation & Preprocessing

This notebook covers data loading, corpus statistics, vocabulary building, and query split analysis for the **AILA 2019 Legal Information Retrieval Benchmark**.

In [ ]:
# Cell 1: Imports & Setup
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_loader import AILADataLoader
from src.preprocessing import Vocabulary, clean_text

In [ ]:
# Cell 2: Load Raw AILA 2019 Dataset
data_dir = os.path.join(project_root, 'data', 'raw', 'AILA2019')
loader = AILADataLoader(data_dir)

queries = loader.load_queries()
casedocs = loader.load_case_docs()
qrels = loader.load_qrels_priorcases()

print(f'Loaded {len(queries)} queries.')
print(f'Loaded {len(casedocs)} legal case documents.')
print(f'Loaded relevance judgments for {len(qrels)} queries.')

In [ ]:
# Cell 3: Data Splits Integrity Check
train_qids, val_qids, test_qids = loader.get_query_splits(queries, qrels)
print(f'Train queries count: {len(train_qids)} | QIDs: {train_qids}')
print(f'Val queries count:   {len(val_qids)} | QIDs: {val_qids}')
print(f'Test queries count:  {len(test_qids)} (Q11-Q50)')

In [ ]:
# Cell 4: Vocabulary Construction (Training Split Only)
train_texts = [queries[q] for q in train_qids if q in queries]
for qid in train_qids:
    for doc_id in qrels.get(qid, set()):
        if doc_id in casedocs:
            train_texts.append(casedocs[doc_id])

vocab = Vocabulary()
vocab.build_vocab(train_texts, max_vocab_size=30000, min_freq=2)
print(f'Vocabulary size built strictly on train split: {len(vocab)} words')